# P2 Project Reimplementation — Natural Gradient Boosting

This notebook is the primary reproducible P2 workflow. The canonical implementation is the visible source tree at `code/modeling/p2_source/`. A small project-local `ngboost` facade preserves the same import style used by the earlier modeling workflow. The external PyPI `ngboost` package is not installed here; historical library outputs are used only as labelled baselines.


## Reproducibility contract

Seed 42 is fixed. The chronological 251/77/46 match split is preserved. The evaluator derives the 128 MD1 pre-match and 189 MD1 live feature contracts from the data-pipeline audit and appends the frozen 18-feature P1 representation, yielding 146 pre-match and 207 live inputs. Hyperparameter selection, Platt calibration, and uncertainty scaling use validation data only.


## 1. Locate the repository


In [ ]:
from pathlib import Path
import os, sys, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

candidates = [
    Path('/content/ML-Project-Football-Forecasting'),
    Path('/content/drive/MyDrive/ml_project'),
    Path('/content/drive/MyDrive/ML-Project-Football-Forecasting'),
    Path.cwd(),
    Path.cwd().parent,
]
required = Path('code/data_pipeline/results/Premier_League_2015_16_DATA_ONLY_FULL_REAL/data/gold/gold_prematch.parquet')
PROJECT_ROOT = next((p for p in candidates if (p / required).exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Repository not found; set PROJECT_ROOT manually.')
PROJECT_ROOT = PROJECT_ROOT.resolve()
MODELING = PROJECT_ROOT / 'code/modeling'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.version)


## 2. Install the P2 runtime

The requirements file is deliberately minimal: it contains only packages imported by the P2 source, evaluator, tests, and Parquet reader. The external `ngboost` package is intentionally absent. Pip output is left visible so any resolver failure names the problematic dependency directly.


In [ ]:
req = MODELING / 'requirements_colab_p2_reimplementation.txt'
print(req.read_text())
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check',
    '-r', str(req)
])


## 3. Verify the visible implementation

The verifier does not reconstruct an archive or rewrite source. It checks that `p2_source/` contains 28 Python files, verifies the exact committed Git tree for that directory, compiles every source file, and confirms the repository-local `ngboost` facade imports successfully. Any mismatch stops the run.


In [ ]:
assert (MODELING / 'p2_source').is_dir()
print('Visible P2 Python files:', len(list((MODELING / 'p2_source').rglob('*.py'))))
subprocess.check_call([sys.executable, str(MODELING / 'prepare_p2_reimplementation.py')])


## 4. Confirm runtime provenance

This must print a module path inside `code/modeling/ngboost`, proving that the P2 workflow is using the repository implementation rather than a site-packages installation.


In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(MODELING)
probe = (
    "import ngboost,inspect; "
    "print('version=',ngboost.__version__); "
    "print('module=',inspect.getfile(ngboost)); "
    "from ngboost import NGBClassifier,NGBRegressor"
)
subprocess.check_call([sys.executable, '-c', probe], env=env)


## 5. Mathematical and API tests

The tests cover deterministic fitting, sklearn clone/Pipeline behavior, categorical probability normalization, finite-difference Normal gradients, Fisher symmetry and positive-definiteness, direct natural-gradient solve agreement, multivariate covariance positive-definiteness, serialization, and distinct natural-versus-Euclidean training paths.


In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pytest', '-q',
    str(MODELING / 'test_p2_reimplementation.py')
], env=env)


## 6. Task C — pre-match outcome

The classifier predicts Away/Draw/Home probabilities. Candidate selection uses validation RPS. Evidence includes raw and validation-only Platt calibrated RPS, log loss, Brier, ECE, accuracy, macro/per-class precision-recall-F1, confusion matrices, reliability bins, row-level predictions, and ten worst cases.


## 7. Task R — pre-match margin

The Normal regressor predicts final home-minus-away margin. Validation NLL selects the candidate; validation residuals determine scale correction. Evidence includes MAE, RMSE, Pearson correlation, raw/calibrated NLL, 50/80/95% coverage and widths, predictive intervals, row predictions, and ten worst errors.


## 8. Task L — causal live forecasting

Each snapshot uses only information available by that cutoff. Live classification and margin regression are compared against frozen P2 pre-match predictions at every minute. The runner writes aggregate, minute-level, and phase-level metrics; calibration evidence; row predictions; worst cases; and live-versus-frozen curves.


## 9. Multivariate Gaussian paper experiment

The paper-aligned experiment predicts a conditional bivariate Gaussian over home and away goals. Candidate selection uses validation joint NLL. Evidence includes joint NLL, Energy Score, marginal/joint 80% coverage, covariance eigenvalues, predicted correlation, validation-only covariance-dispersion scaling, and derived H/D/A probabilities.


## 10. Compute and memory

Training wall time is measured directly. Peak RSS is sampled during fitting in a background thread. Prediction latency is measured repeatedly and summarized with p50/p95/p99 batch times.


## 11. Execute the complete held-out evaluation

This is the expensive cell. Run it only after dependency installation, source verification, provenance checking, and tests succeed.


In [ ]:
runner = MODELING / 'run_p2_reimplementation_final.py'
subprocess.check_call([
    sys.executable, str(runner),
    '--mode', 'full',
    '--project-root', str(PROJECT_ROOT)
], env=env)


## 12. Unified P2 versus official-library comparison

Historical PyPI NGBoost results are read only after P2 finishes and are explicitly labelled as library baselines.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
OUT = PROJECT_ROOT / 'code/modeling/outputs/p2_reimplementation_full'
TAB = OUT / 'tables'
display(pd.read_csv(TAB / 'p2_vs_official_library.csv'))


## 13. Review generated Task C/R/L and multivariate evidence


In [ ]:
for name in [
    'p2_task_c_metrics.csv',
    'p2_task_r_metrics.csv',
    'p2_task_l_outcome_metrics.csv',
    'p2_task_l_margin_metrics.csv',
    'p2_task_l_outcome_by_phase.csv',
    'p2_task_l_margin_by_phase.csv',
    'p2_multivariate_metrics.csv',
    'p2_compute_and_peak_memory.csv',
]:
    p = TAB / name
    if p.exists():
        print('\n###', name)
        display(pd.read_csv(p))


## 14. Completion checklist and report insert

Do not report P2 numerical claims until this notebook completes. Review the evidence-derived checklist and generated report material after the run.


In [ ]:
print((OUT / 'p2_completion_checklist.json').read_text())
display(Markdown((OUT / 'P2_REPORT_INSERT.md').read_text(encoding='utf-8')))


## Interpretation and limitations

The held-out set contains 46 matches, so small ranking differences are uncertain. Snapshot rows from a match are repeated observations, not independent matches. Gaussian margin and score assumptions can remain imperfectly calibrated. The official-library comparison is descriptive evidence, not proof that the two implementations are mathematically identical.
